In [1]:

import numpy as np
import numpy.typing as npt
import pickle
from itertools import product
from typing import Optional

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector, Parameter

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler


from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from hubo_qaoa.utils.parameterise_circuit import parameterise_circuit
from hubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit

from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples


In [2]:
backend_options = dict(
    method='matrix_product_state',
    matrix_product_state_max_bond_dimension='32', 
    device='CPU',
    precision='single',
    basis_gates = ['rx', 'ry', 'rz', 'cx']
)
# fake_fez = FakeFez()
backend = AerSimulator(**backend_options)
sampler = Sampler.from_backend(backend)

filename: str = 'test_N2_W2'
shots: int = 4000
copy_numbers = [1,1]

rng = np.random.default_rng()

data_file = '/lustre/scratch127/qpg/jc59/new_hubo_formulation/circuit_depths/results.couplingall.precompute.0.pkl'
with open(data_file, 'rb') as f:
    res = pickle.load(f)
cost_circuit = parameterise_circuit(res[filename]['rzz']['circuit'], parameter=Parameter('γ'))
num_qubits: int = cost_circuit.num_qubits    
    
filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
full_hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)


eta = 1
eps = 0.005

def normalisation(energies: npt.NDArray, beta_T: float) -> float:
    return sum([np.exp(-beta_T * E) for E in energies])

def boltzmann(energies: npt.NDArray, Z: float, beta_T: float) -> npt.NDArray:
    return np.exp(- beta_T * energies) / Z

def bias(boltzmanns: npt.NDArray, q: int, samples: list[str]) -> float:
    return sum([boltzmanns[i] * (-1 if samples[i][q] == '1' else 1) for i in range(len(samples))])

def get_biases(samples: list[str], energies: npt.NDArray, beta_T: float) -> npt.NDArray:
    Z = normalisation(energies, beta_T)
    boltzmanns = boltzmann(energies, Z, beta_T)
    return np.array([bias(boltzmanns, q, samples) for q in range(num_qubits)][::-1])

def get_angles(samples: list[str], energies: npt.NDArray, beta_T: float) -> npt.NDArray:
    biases = get_biases(samples, energies, beta_T)
    probabilities = 0.5 * (1 - eta * biases)
    probabilities[probabilities < eps] = eps
    probabilities[probabilities > 1 - eps] = 1 - eps
    
    angles = 2 * np.arcsin(np.sqrt(probabilities))
    return angles


def iteration(qc, angles, beta_T, history):
    sample_circuit = qc.assign_parameters(angles, inplace=False)
    sampler_job = sampler.run([sample_circuit],shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    
    evals = evaluate_sparse_pauli_samples(counts.keys(), full_hamiltonian * norm)
    samples, energies = [], []
    for idx, (sample, count) in enumerate(counts.items()):
        samples.extend(count * [sample])
        energies.extend(count * [evals[idx]])
    total_energy = np.mean(energies)
    
    
    history.append([samples, energies, total_energy])
    new_angles = get_angles(samples, np.array(energies), beta_T)
    return new_angles



prob = 1 / 2
theta = 2 * np.arcsin(np.sqrt(prob))
init_angles = theta * np.ones((num_qubits,))



def warm_start(p: int, delta_b: float, delta_g: float, circ: Optional[QuantumCircuit]=None):
    phis = ParameterVector('ϕ', num_qubits)
    fixed_qc, circuit = get_LR_qaoa_circuit(p, delta_b, delta_g, num_qubits, cost_circuit, circ, phis, True)
    fixed_qc.measure_all(add_bits=False)
    history = []
    angles = [init_angles]
    iters = 10

    for i in range(iters):
        beta_T = (i ** 2) * 0.9 / (iters - 1)**2 + 0.1
        angles.append(iteration(fixed_qc, angles[-1], beta_T, history))
        
    energy = history[-1][2]
    samples = (history[0][0], history[iters // 2][0], history[-1][0])
    print(f'delta_b:{np.round(delta_b, 2)}, delta_g:{np.round(delta_g, 2)}, p:{p}, energy:{np.round(energy, 2)}')
    print(angles)
    return energy, samples, circuit
        
delta_b_fixed = 0.45
delta_g_fixed = 0.26

rescaling = 10** np.linspace(-1, 0.5, 6)
ps = sorted(set([int(p) for p in np.logspace(0, 2.5, 11, base=10)]))

energies = np.zeros((len(ps), len(rescaling)))
samples_dict = {}



Keeping constraints at times: [0]


In [3]:
circuit = None
e, samples, circuit = warm_start(1, delta_b_fixed * 1, delta_g_fixed * 1, circuit)

Mixer params: ParameterView([Parameter(β), ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3])])
Mixer bind dict: {Parameter(β): ParameterVectorElement(β[0])}
12:06:32 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 15
delta_b:0.45, delta_g:0.26, p:1, energy:0.07
[array([1.57079633, 1.57079633, 1.57079633, 1.57079633]), array([1.58401973, 1.57108368, 1.56437624, 1.55513186]), array([1.58010272, 1.55885061, 1.57561922, 1.53990186]), array([1.59777566, 1.53629221, 1.55347842, 1.54774992]), array([1.651176  , 1.49171342, 1.488093  , 1.48616026]), array([1.83551982, 1.31048374, 1.30702427, 1.30843491]), array([2.33670811, 0.80640868, 0.8053301 , 0.8064197 ]), array([2.9943828 , 0.14911801, 0.14672136, 0.14880172]), array([3.00005318, 0.14153947, 0.14153947, 0.14153947]), array([3.00005318, 0.14153947, 0.14153947, 0.14153947]), array([3.00005318, 0.14153947, 0.14153947, 0.14153947])]


In [4]:
phis = ParameterVector('ϕ', num_qubits)
fixed_qc, circuit = get_LR_qaoa_circuit(1, delta_b_fixed, delta_g_fixed, num_qubits, cost_circuit, None, phis, True)

Mixer params: ParameterView([Parameter(β), ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3])])
Mixer bind dict: {Parameter(β): ParameterVectorElement(β[0])}
12:06:33 - hubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 15


In [5]:
fixed_qc.draw(fold=-1)

┌──────────┐                                                                                                                                 ┌───────────┐┌───────────┐┌──────────┐ ░ ┌─┐         
q_0: ┤ Ry(ϕ[0]) ├──■─────────────────────■────────────■──────────────────────────────────────────────■─────────────────────────────────■──────────┤ Ry(-ϕ[0]) ├┤ Rz(-0.45) ├┤ Ry(ϕ[0]) ├─░─┤M├─────────
     ├──────────┤┌─┴─┐┌───────────┐┌───┐ │ZZ(-0.325)  │                                   ┌───┐    ┌─┴─┐                 ┌───────────┐ │          ├───────────┤└┬──────────┤└──────────┘ ░ └╥┘┌─┐      
q_1: ┤ Ry(ϕ[1]) ├┤ X ├┤ Rz(0.325) ├┤ X ├─■────────────┼───────────■───────────────────────┤ X ├────┤ X ├─────■───────────┤ Ry(-ϕ[1]) ├─┼──────────┤ Rz(-0.45) ├─┤ Ry(ϕ[1]) ├─────────────░──╫─┤M├──────
     ├──────────┤└───┘└───────────┘└─┬─┘              │ZZ(0.585)  │                       └─┬─┘┌───┴───┴───┐ │           ├───────────┤ │          └┬──────────┤ └──────────┘             ░  ║ └╥┘┌─┐   
q_2: ┤ Ry(ϕ[2]) ├────────────────────■────────────────■───────────┼───────────■─────────────■──┤ Ry(-ϕ[2]) ├─┼───────────┤ Rz(-0.45) ├─┼───────────┤ Ry(ϕ[2]) ├──────────────────────────░──╫──╫─┤M├───
     ├──────────┤                                                 │ZZ(0.325)  │ZZ(-0.325)      └───────────┘ │ZZ(-0.325) └───────────┘ │ZZ(0.325) ┌┴──────────┤┌───────────┐┌──────────┐ ░  ║  ║ └╥┘┌─┐
q_3: ┤ Ry(ϕ[3]) ├─────────────────────────────────────────────────■───────────■──────────────────────────────■─────────────────────────■──────────┤ Ry(-ϕ[3]) ├┤ Rz(-0.45) ├┤ Ry(ϕ[3]) ├─░──╫──╫──╫─┤M├
     └──────────┘                                                                                                                                 └───────────┘└───────────┘└──────────┘ ░  ║  ║  ║ └╥┘
c: 4/═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╩══╩══╩══╩═
                                                                                                                                                                                            0  1  2  3

In [6]:
sample_circuit = fixed_qc.assign_parameters(init_angles, inplace=False)
sample_circuit.draw(fold=-1)

┌─────────┐                                                                                                                                 ┌──────────┐┌───────────┐┌─────────┐ ░ ┌─┐         
q_0: ┤ Ry(π/2) ├──■─────────────────────■────────────■─────────────────────────────────────────────■─────────────────────────────────■───────────┤ Ry(-π/2) ├┤ Rz(-0.45) ├┤ Ry(π/2) ├─░─┤M├─────────
     ├─────────┤┌─┴─┐┌───────────┐┌───┐ │ZZ(-0.325)  │                                   ┌───┐   ┌─┴─┐                  ┌──────────┐ │          ┌┴──────────┤└┬─────────┬┘└─────────┘ ░ └╥┘┌─┐      
q_1: ┤ Ry(π/2) ├┤ X ├┤ Rz(0.325) ├┤ X ├─■────────────┼───────────■───────────────────────┤ X ├───┤ X ├─────■────────────┤ Ry(-π/2) ├─┼──────────┤ Rz(-0.45) ├─┤ Ry(π/2) ├─────────────░──╫─┤M├──────
     ├─────────┤└───┘└───────────┘└─┬─┘              │ZZ(0.585)  │                       └─┬─┘┌──┴───┴───┐ │           ┌┴──────────┤ │          └┬─────────┬┘ └─────────┘             ░  ║ └╥┘┌─┐   
q_2: ┤ Ry(π/2) ├────────────────────■────────────────■───────────┼───────────■─────────────■──┤ Ry(-π/2) ├─┼───────────┤ Rz(-0.45) ├─┼───────────┤ Ry(π/2) ├──────────────────────────░──╫──╫─┤M├───
     ├─────────┤                                                 │ZZ(0.325)  │ZZ(-0.325)      └──────────┘ │ZZ(-0.325) └───────────┘ │ZZ(0.325)  ├─────────┴┐┌───────────┐┌─────────┐ ░  ║  ║ └╥┘┌─┐
q_3: ┤ Ry(π/2) ├─────────────────────────────────────────────────■───────────■─────────────────────────────■─────────────────────────■───────────┤ Ry(-π/2) ├┤ Rz(-0.45) ├┤ Ry(π/2) ├─░──╫──╫──╫─┤M├
     └─────────┘                                                                                                                                 └──────────┘└───────────┘└─────────┘ ░  ║  ║  ║ └╥┘
c: 4/════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╩══╩══╩══╩═
                                                                                                                                                                                         0  1  2  3

In [7]:
sampler_job = sampler.run([sample_circuit],shots=shots)
sampler_result = sampler_job.result()
counts = sampler_result[0].data.c.get_counts()

In [8]:
counts

{'0001': 727,
 '1000': 188,
 '0010': 207,
 '0111': 193,
 '1100': 196,
 '1101': 181,
 '0110': 199,
 '0011': 187,
 '1001': 192,
 '0000': 127,
 '1111': 83,
 '1110': 752,
 '1011': 296,
 '0100': 262,
 '1010': 116,
 '0101': 94}